In [1]:
import pandas as pd

In [2]:
# Read in the variant metadata for the baseline and each attack.
baseline_metadata = pd.read_csv("./data/raw/baseline/variants_data.csv")
rwa_attack_metadata = pd.read_csv("./data/raw/attack_rwa/variants_data.csv")
rwb_attack_metadata = pd.read_csv("./data/raw/attack_rwb/variants_data.csv")
rwc_attack_metadata = pd.read_csv("./data/raw/attack_rwc/variants_data.csv")

In [3]:
# Add a column for Scenario to the baseline metadata.  It should be "baseline"
baseline_metadata['Scenario'] = 'baseline'

# Set Scenario to "A" for the attack 1 metadata
rwa_attack_metadata['Scenario'] = 'attack_rwa'

# Set Scenario to "B" for the attack 2 metadata
rwb_attack_metadata['Scenario'] = 'attack_rwb'

# Set Scenario to "C" for the attack 3 metadata
rwc_attack_metadata['Scenario'] = 'attack_rwc'

In [4]:
# Combine all metadata into a single DataFrame
all_metadata = pd.concat([baseline_metadata, rwa_attack_metadata, rwb_attack_metadata, rwc_attack_metadata], ignore_index=True)

In [ ]:
from sklearn.model_selection import train_test_split

remainder, training = train_test_split(all_metadata, test_size=0.10, random_state=42, stratify=all_metadata['Scenario'])

discard, test = train_test_split(remainder, test_size=0.10, random_state=42, stratify=remainder['Scenario'])

In [5]:
def prepare_metadata(variants_df):
    # Add a column for Month to the metadata, it is the first token of the Variant_name
    variants_df['Month'] = variants_df['Variant_name'].str.split('_').str[0]
    # Add a column for Time_offset to the metadata, it is the second token of the Variant_name
    variants_df['Time_offset'] = variants_df['Variant_name'].str.split('_').str[1]
    # Convert the Month column to the month name
    variants_df['Month'] = pd.to_datetime(variants_df['Month'], format='%m').dt.strftime('%B')
    # Set the Description column to the Run_Batch column + Month name + Time_offset
    variants_df['Description'] = variants_df['Run_Batch'] + ' ' + variants_df['Month'] + ' Time Offset:' + variants_df['Time_offset']
    variants_df['data_file'] = variants_df.apply(lambda row: f"./data/raw/{row['Scenario']}/fsw_data_{row['Variant_name']}.csv", axis=1)
    return variants_df.drop(columns=['description', '_id'])

In [ ]:
all_metadata = prepare_metadata(all_metadata)

In [8]:
all_metadata = all_metadata.drop(columns=['Simulation', 'FSW', 'Simulation_rows'])

In [11]:
all_metadata.to_csv('C:/Users/JoeDavis/OneDrive - BigBear.ai/Desktop/Projects/Space Innovation/SCAD/data/raw/all_metadata.csv', index=False)

In [12]:
baseline_metadata = all_metadata[all_metadata['Scenario'] == 'baseline']
rwa_attack_metadata = all_metadata[all_metadata['Scenario'] == 'attack_rwa']
rwb_attack_metadata = all_metadata[all_metadata['Scenario'] == 'attack_rwb']
rwc_attack_metadata = all_metadata[all_metadata['Scenario'] == 'attack_rwc']


In [13]:
baseline_metadata.to_csv('C:/Users/JoeDavis/OneDrive - BigBear.ai/Desktop/Projects/Space Innovation/SCAD/data/raw/baseline/baseline_metadata.csv', index=False)

In [14]:
rwa_attack_metadata.to_csv('C:/Users/JoeDavis/OneDrive - BigBear.ai/Desktop/Projects/Space Innovation/SCAD/data/raw/attack_rwa/rwa_attack_metadata.csv', index=False)
rwb_attack_metadata.to_csv('C:/Users/JoeDavis/OneDrive - BigBear.ai/Desktop/Projects/Space Innovation/SCAD/data/raw/attack_rwb/rwb_attack_metadata.csv', index=False)
rwc_attack_metadata.to_csv('C:/Users/JoeDavis/OneDrive - BigBear.ai/Desktop/Projects/Space Innovation/SCAD/data/raw/attack_rwc/rwc_attack_metadata.csv', index=False)

In [ ]:
training_metadata = prepare_metadata(training)
test_metadata = prepare_metadata(test)

In [17]:
import os

In [18]:
def get_data_and_label(variants_df):
    scenario_map = {
        'baseline': 'baseline',
        'attack_rwa': "RWA",
        'attack_rwb': "RWB",
        'attack_rwc': "RWC"
    }
    df = pd.DataFrame()
    # For each datafile, load the data and append it to the training_df
    for index, row in variants_df.iterrows():
        data_file = row['data_file']
        if os.path.exists(data_file):
            data = pd.read_csv(data_file)
            data['Scenario'] = row['Scenario']
            data['Run_Batch_Variant'] = data['Run_Batch'] + '_' + data['Variant']
            # Always Label the first 302 rows as baseline and the rest as the Scenario
            #map the Scenario to the result_label using the scenario_map
            data['result_label'] = data['Scenario'].map(scenario_map)
            data.loc[data.index < 302, "result_label"] = "baseline"
            df = pd.concat([df, data], ignore_index=True)
    return df

In [19]:
baseline_df = get_data_and_label(baseline_metadata)

In [21]:
baseline_df = baseline_df.drop(columns=['_id'])

In [22]:
# Group the FSW data by Variant
baseline_grouped = baseline_df.groupby("Variant")
for variant, group in baseline_grouped:
    group.to_csv(f"C:/Users/JoeDavis/OneDrive - BigBear.ai/Desktop/Projects/Space Innovation/SCAD/data/raw/baseline/fsw_data_{variant}.csv", index=False)

In [23]:
rwa_df = get_data_and_label(rwa_attack_metadata)
rwb_df = get_data_and_label(rwb_attack_metadata)
rwc_df = get_data_and_label(rwc_attack_metadata)

In [24]:
rwa_df = rwa_df.drop(columns=['_id'])
rwb_df = rwb_df.drop(columns=['_id'])
rwc_df = rwc_df.drop(columns=['_id'])

In [25]:
rwa_grouped = rwa_df.groupby("Variant")
for variant, group in rwa_grouped:
    group.to_csv(f"C:/Users/JoeDavis/OneDrive - BigBear.ai/Desktop/Projects/Space Innovation/SCAD/data/raw/attack_rwa/fsw_data_{variant}.csv", index=False)

In [26]:
rwb_grouped = rwb_df.groupby("Variant")
for variant, group in rwb_grouped:
    group.to_csv(f"C:/Users/JoeDavis/OneDrive - BigBear.ai/Desktop/Projects/Space Innovation/SCAD/data/raw/attack_rwb/fsw_data_{variant}.csv", index=False)

In [27]:
rwc_grouped = rwc_df.groupby("Variant")
for variant, group in rwc_grouped:
    group.to_csv(f"C:/Users/JoeDavis/OneDrive - BigBear.ai/Desktop/Projects/Space Innovation/SCAD/data/raw/attack_rwc/fsw_data_{variant}.csv", index=False)

In [ ]:
training_df = get_data_and_label(training)
test_df = get_data_and_label(test)

In [ ]:
training_df.head()

In [ ]:
def transform_data(df, scaler):
    # Drop the unnecessary columns
    df = df.drop(columns=['_id', 'Variant', 'Run_Batch', 'date_created', 'date_loaded', 'Scenario'])

    # Encode the result_label column, baseline as 0, attack_1 as 1
    #df['result_label_dummy'] = df['result_label']
    #df = pd.get_dummies(df, columns=['result_label_dummy'], drop_first=False, prefix='result_label') 
    # Encode the Mode column with one-hot encoding
    #df['Mode_dummy'] = df['Mode()']
    #df = pd.get_dummies(df, columns=['Mode_dummy'], drop_first=False, prefix='Mode')

    # Convert UTC to datetime
    df['UTC'] = pd.to_datetime(df['UTC'], utc=True)

    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.difference(['UTC'])
    # If scaler is not fitted, fit it
    if not hasattr(scaler, 'mean_'):
        scaler.fit(df[numeric_cols])
    df[numeric_cols] = scaler.transform(df[numeric_cols])

    
    return df

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

training_df_transformed = transform_data(training_df, scaler)

In [ ]:
# Create a random forest classifier and train it on the training data
from sklearn.ensemble import RandomForestClassifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(training_df_transformed.drop(columns=['UTC', 'Run_Batch_Variant', 'result_label', 'FswTime(sec)']), training_df_transformed['result_label'])

In [ ]:
test_df_transformed = transform_data(test_df, scaler)

In [ ]:
# Evaluate the model on the test data
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import numpy as np

predictions = rf_classifier.predict(test_df_transformed.drop(columns=['UTC', 'result_label', 'Run_Batch_Variant', 'FswTime(sec)']))
print(classification_report(test_df_transformed['result_label'], predictions))
print("Accuracy:", accuracy_score(test_df_transformed['result_label'], predictions))

#Create a confusion matrix
conf_matrix = confusion_matrix(test_df_transformed['result_label'], predictions)
print("Confusion Matrix:\n", conf_matrix)

# visualize the confusion matrix
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 7))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['RWA', "RWB", "RWC", 'Baseline'], yticklabels=['RWA', "RWB", "RWC", 'Baseline'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# Combine all metadata into a single DataFrame
all_metadata = pd.concat([baseline_metadata, rwa_attack_metadata], ignore_index=True)

remainder, training = train_test_split(all_metadata, test_size=0.10, random_state=42, stratify=all_metadata['Scenario'])
discard, test = train_test_split(remainder, test_size=0.10, random_state=42, stratify=remainder['Scenario'])

training_metadata = prepare_metadata(training)
test_metadata = prepare_metadata(test)

training_df = get_data_and_label(training)
test_df = get_data_and_label(test)

scaler = StandardScaler()
training_df_transformed = transform_data(training_df, scaler)
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(training_df_transformed.drop(columns=['UTC', 'Run_Batch_Variant', 'result_label', 'FswTime(sec)']), training_df_transformed['result_label'])

test_df_transformed = transform_data(test_df, scaler)
predictions = rf_classifier.predict(test_df_transformed.drop(columns=['UTC', 'result_label', 'Run_Batch_Variant', 'FswTime(sec)']))
print(classification_report(test_df_transformed['result_label'], predictions))
print("Accuracy:", accuracy_score(test_df_transformed['result_label'], predictions))

#Create a confusion matrix
conf_matrix = confusion_matrix(test_df_transformed['result_label'], predictions)
print("Confusion Matrix:\n", conf_matrix)

# visualize the confusion matrix
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 7))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['RWA', 'Baseline'], yticklabels=['RWA', 'Baseline'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()